# Module 17 - Retrieval-augmented generation

Use this notebook after `tests/test_rag.py` is passing. The notebook builds a small RAG pipeline in layers: chunks, embeddings, vector search, prompt assembly, retrieval, then generation through a backend.

The deliverable is the RAG postmortem: what you indexed, what retrieved well, where retrieval failed, and what you would improve before using this as part of the assistant.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from g2c.inference import Backend, BackendInfo, InferenceResult, load_selected_backend
from g2c.notebook_extras.sampling import printable
from g2c.rag import (
    DEFAULT_INSTRUCTION,
    DEFAULT_OLLAMA_EMBED_MODEL,
    DEFAULT_SYSTEM,
    Chunk,
    DenseRetriever,
    HashEmbedder,
    NumpyVectorStore,
    OllamaEmbedder,
    RAGPipeline,
    RetrievedChunk,
    assemble_rag_prompt,
    chunk_text,
    cosine_similarity,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

/Users/colkitt/sith/toys/courses/g2c


Run the RAG tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 17 TODOs in `g2c/rag/`.

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rag.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 17 RAG tests are not passing yet."

........................................................................ [ 50%]
......................................................................   [100%]



## Display helpers

In [3]:
def short(text: Any, limit: int = 220) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def show_chunks(chunks: list[Chunk], *, limit: int = 8) -> None:
    rows = []
    for i, chunk in enumerate(chunks[:limit], start=1):
        rows.append(
            {
                "i": i,
                "source": chunk.source,
                "span": f"{chunk.start}:{chunk.end}",
                "chars": len(chunk.text),
                "text": short(chunk.text, 120),
            }
        )
    display(Markdown(markdown_table(rows, ["i", "source", "span", "chars", "text"])))
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")


def show_retrieved(results: list[RetrievedChunk]) -> None:
    rows = []
    for r in results:
        rows.append(
            {
                "rank": r.rank,
                "score": f"{r.score:.3f}",
                "source": r.chunk.source,
                "text": short(r.chunk.text, 180),
            }
        )
    display(Markdown(markdown_table(rows, ["rank", "score", "source", "text"])))


def show_rag_answer(answer) -> None:
    print(printable(answer.answer))
    print()
    print("Sources:")
    show_retrieved(answer.retrieved)
    print("metadata:", answer.metadata)


## A tiny corpus

The first pass uses a deliberately small in-memory corpus so the moving parts are visible. Later cells swap in course docs and optional Ollama embeddings.

In [4]:
documents = {
    "cities.md": (
        "Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\n"
        "Paris is the capital of France. The Seine river runs through the city.\n\n"
        "Tokyo is the capital of Japan and one of the largest metropolitan areas in the world."
    ),
    "fruit.md": (
        "Bananas are yellow fruit rich in potassium.\n\n"
        "Apples grow on trees and can be red, green, or yellow.\n\n"
        "Oranges are citrus fruit and are often used for juice."
    ),
    "course.md": (
        "Module 16 introduces inference backends and ProdLM.\n\n"
        "Module 17 introduces retrieval augmented generation over an external corpus.\n\n"
        "Module 18 adds tools so the assistant can act outside text generation."
    ),
}

for source, text in documents.items():
    print(source, len(text), "chars")


cities.md 248 chars
fruit.md 155 chars
course.md 201 chars


## Exercise 1 - Chunk documents

`chunk_text` turns each source document into overlapping retrievable slices. The key parameters are `chunk_size` and `chunk_overlap`.

In [5]:
all_chunks: list[Chunk] = []
for source, text in documents.items():
    all_chunks.extend(
        chunk_text(
            text,
            source=source,
            chunk_size=120,
            chunk_overlap=30,
            metadata={"corpus": "toy"},
        )
    )

print("chunks:", len(all_chunks))
show_chunks(all_chunks)

chunks: 7


| i | source | span | chars | text |
| --- | --- | --- | --- | --- |
| 1 | cities.md | 0:120 | 120 | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of ... |
| 2 | cities.md | 90:210 | 120 | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of ... |
| 3 | cities.md | 180:248 | 68 | tal of Japan and one of the largest metropolitan areas in the world. |
| 4 | fruit.md | 0:120 | 120 | Bananas are yellow fruit rich in potassium.\n\nApples grow on trees and can be red, green, or yellow.\n\nOranges are ... |
| 5 | fruit.md | 90:155 | 65 | r yellow.\n\nOranges are citrus fruit and are often used for juice. |
| 6 | course.md | 0:120 | 120 | Module 16 introduces inference backends and ProdLM.\n\nModule 17 introduces retrieval augmented generation over an ex... |
| 7 | course.md | 90:201 | 111 | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |

In [6]:
# Inspect the overlap directly on the first document.
city_chunks = [chunk for chunk in all_chunks if chunk.source == "cities.md"]
show_chunks(city_chunks)

if len(city_chunks) >= 2:
    previous = city_chunks[0]
    current = city_chunks[1]
    overlap = previous.end - current.start
    print("actual overlap:", overlap)
    print("previous suffix:", repr(previous.text[-overlap:]))
    print("current prefix:", repr(current.text[:overlap]))

| i | source | span | chars | text |
| --- | --- | --- | --- | --- |
| 1 | cities.md | 0:120 | 120 | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of ... |
| 2 | cities.md | 90:210 | 120 | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of ... |
| 3 | cities.md | 180:248 | 68 | tal of Japan and one of the largest metropolitan areas in the world. |

actual overlap: 30
previous suffix: '\nParis is the capital of Franc'
current prefix: '\nParis is the capital of Franc'


Try a few chunk sizes and watch the number of chunks change. Smaller chunks retrieve more specifically; larger chunks carry more context but blur the embedding.

In [7]:
chunk_size_sweep = []
for size in [60, 100, 160, 240]:
    overlap = max(0, size // 5)
    n_chunks = sum(
        len(chunk_text(text, source=source, chunk_size=size, chunk_overlap=overlap))
        for source, text in documents.items()
    )
    chunk_size_sweep.append({"chunk_size": size, "overlap": overlap, "chunks": n_chunks})

display(Markdown(markdown_table(chunk_size_sweep, ["chunk_size", "overlap", "chunks"])))

| chunk_size | overlap | chunks |
| --- | --- | --- |
| 60 | 12 | 12 |
| 100 | 20 | 8 |
| 160 | 32 | 5 |
| 240 | 48 | 4 |

## Exercise 2 - Hash embeddings

`HashEmbedder` is a toy lexical embedder. It is not semantic, but it makes the same vector-store math work without an external embedding model.

In [8]:
hash_embedder = HashEmbedder(dim=512, ngram_range=(3, 5), seed=0)
chunk_vectors = hash_embedder.embed([chunk.text for chunk in all_chunks])

print("vectors:", chunk_vectors.shape, chunk_vectors.dtype)
print("first five row norms:", np.linalg.norm(chunk_vectors[:5], axis=1))

vectors: (7, 512) float32
first five row norms: [0.99999994 1.         1.         1.         1.        ]


In [9]:
probe_texts = [
    "Madrid is the capital of Spain.",
    "Madrid is Spain's capital city.",
    "What is the capital of Spain?",
    "Bananas are a yellow fruit.",
]
probe_vectors = hash_embedder.embed(probe_texts)

rows = []
for i, a in enumerate(probe_texts):
    for j, b in enumerate(probe_texts):
        if j <= i:
            continue
        rows.append(
            {
                "a": short(a, 45),
                "b": short(b, 45),
                "cosine": f"{cosine_similarity(probe_vectors[i], probe_vectors[j]):.3f}",
            }
        )

display(Markdown(markdown_table(rows, ["a", "b", "cosine"])))

| a | b | cosine |
| --- | --- | --- |
| Madrid is the capital of Spain. | Madrid is Spain's capital city. | 0.667 |
| Madrid is the capital of Spain. | What is the capital of Spain? | 0.811 |
| Madrid is the capital of Spain. | Bananas are a yellow fruit. | 0.127 |
| Madrid is Spain's capital city. | What is the capital of Spain? | 0.506 |
| Madrid is Spain's capital city. | Bananas are a yellow fruit. | 0.123 |
| What is the capital of Spain? | Bananas are a yellow fruit. | 0.144 |

## Exercise 3 - Build and search a vector store

A flat vector store keeps chunks and vectors in matching order. Search is one dot product per chunk plus a top-k selection.

In [10]:
store = NumpyVectorStore(dim=hash_embedder.dim)
store.add(all_chunks, chunk_vectors)
print(store)

query = "What city is the capital of Spain?"
query_vector = hash_embedder.embed([query])[0]
raw_results = store.search(query_vector, k=4)

rows = []
for rank, (chunk, score) in enumerate(raw_results, start=1):
    rows.append(
        {
            "rank": rank,
            "score": f"{score:.3f}",
            "source": chunk.source,
            "text": short(chunk.text, 180),
        }
    )

display(Markdown(markdown_table(rows, ["rank", "score", "source", "text"])))

NumpyVectorStore(dim=512, n=7)


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.735 | cities.md | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc |
| 2 | 0.630 | cities.md | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la |
| 3 | 0.336 | cities.md | tal of Japan and one of the largest metropolitan areas in the world. |
| 4 | 0.259 | course.md | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |

In [11]:
query_examples = [
    "What city is the capital of Spain?",
    "Which module adds tools?",
    "What fruit has potassium?",
    "Which river runs through Paris?",
]

for query in query_examples:
    qvec = hash_embedder.embed([query])[0]
    results = store.search(qvec, k=2)
    print("=" * 80)
    print(query)
    for rank, (chunk, score) in enumerate(results, start=1):
        print(f"[{rank}] score={score:.3f} source={chunk.source} text={short(chunk.text, 140)}")

What city is the capital of Spain?
[1] score=0.735 source=cities.md text=Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc
[2] score=0.630 source=cities.md text=\nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la
Which module adds tools?
[1] score=0.391 source=course.md text=ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation.
[2] score=0.283 source=course.md text=Module 16 introduces inference backends and ProdLM.\n\nModule 17 introduces retrieval augmented generation over an externa
What fruit has potassium?
[1] score=0.417 source=fruit.md text=Bananas are yellow fruit rich in potassium.\n\nApples grow on trees and can be red, green, or yellow.\n\nOranges are citrus 
[2] score=0.346 source=fruit.md text=r yellow.\n\nOranges are citrus fruit and are often used for juice.
Which river

## Exercise 4 - Use the retriever abstraction

`DenseRetriever` wires the embedder and vector store into the interface later assistant modules expect: `retrieve(query, k)` returns ranked chunks.

In [12]:
retriever = DenseRetriever(hash_embedder, store)
retrieved = retriever.retrieve("Which module adds tools to the assistant?", k=3)
show_retrieved(retrieved)

| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.511 | course.md | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |
| 2 | 0.332 | cities.md | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc |
| 3 | 0.331 | cities.md | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la |

## Exercise 5 - Assemble a RAG prompt

Prompt assembly is the augmentation step. The retrieved chunks are numbered so the model can cite them.

In [13]:
question = "Which module adds tools to the assistant?"
prompt = assemble_rag_prompt(question, retrieved)
print(prompt.text)

You are a helpful assistant. Answer the user's question using ONLY the context below. Refer to context items by their bracket numbers, e.g. [1], [2].

Context:
[1] (source: course.md)
ted generation over an external corpus.

Module 18 adds tools so the assistant can act outside text generation.

[2] (source: cities.md)
Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.

Paris is the capital of Franc

[3] (source: cities.md)

Paris is the capital of France. The Seine river runs through the city.

Tokyo is the capital of Japan and one of the la

Question: Which module adds tools to the assistant?

If the context does not contain the information needed to answer the question, reply with: "I don't know based on the provided context." Do not invent facts that are not in the context.


In [14]:
empty_prompt = assemble_rag_prompt("What is the population of Pluto?", [])
print(empty_prompt.text)

You are a helpful assistant. Answer the user's question using ONLY the context below. Refer to context items by their bracket numbers, e.g. [1], [2].

Context:

Question: What is the population of Pluto?

If the context does not contain the information needed to answer the question, reply with: "I don't know based on the provided context." Do not invent facts that are not in the context.


## Exercise 6 - End-to-end RAG with a fake backend

Before calling a live model, use a deterministic backend. This confirms retrieval and prompt assembly are wired correctly.

In [15]:
class FakeBackend(Backend):
    def __init__(self, scripted_answer: str = "MOCK ANSWER") -> None:
        self.scripted_answer = scripted_answer
        self.last_prompt: str | None = None
        self.last_kwargs: dict[str, Any] | None = None
        self._info = BackendInfo(name="fake", model_id="fake-rag")

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        self.last_prompt = prompt
        self.last_kwargs = {
            "max_new_tokens": max_new_tokens,
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p,
        }
        return InferenceResult(
            prompt=prompt,
            completion=self.scripted_answer,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(self.scripted_answer.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [16]:
fake_backend = FakeBackend("Module 18 adds tools to the assistant [1].")
fake_pipeline = RAGPipeline(retriever, fake_backend)
fake_answer = fake_pipeline.answer("Which module adds tools to the assistant?", k=3)
show_rag_answer(fake_answer)

print("\nPrompt sent to backend:")
print(fake_backend.last_prompt)


Module 18 adds tools to the assistant [1].

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.511 | course.md | ted generation over an external corpus.\n\nModule 18 adds tools so the assistant can act outside text generation. |
| 2 | 0.332 | cities.md | Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\nParis is the capital of Franc |
| 3 | 0.331 | cities.md | \nParis is the capital of France. The Seine river runs through the city.\n\nTokyo is the capital of Japan and one of the la |

metadata: {'k': 3, 'n_retrieved': 3, 'backend_name': 'fake', 'backend_model_id': 'fake-rag'}

Prompt sent to backend:
You are a helpful assistant. Answer the user's question using ONLY the context below. Refer to context items by their bracket numbers, e.g. [1], [2].

Context:
[1] (source: course.md)
ted generation over an external corpus.

Module 18 adds tools so the assistant can act outside text generation.

[2] (source: cities.md)
Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.

Paris is the capital of Franc

[3] (source: cities.md)

Paris is the capital of France. The Seine river runs through the city.

Tokyo is the capital of Japan and one of the la

Question: Which module adds tools to the assistant?

If the context does not contain the information needed to answer the question, reply with: "I don't know based on the provided context." Do not invent facts that are not in the context.


## Exercise 7 - Index course module docs

The toy corpus is enough to understand the mechanics. A useful RAG pipeline needs a corpus you care about. This cell indexes the course module markdown files with the hash embedder.

In [17]:
DOC_GLOB = "*.md"
DOC_LIMIT = 24
COURSE_CHUNK_SIZE = 1400
COURSE_CHUNK_OVERLAP = 200

module_dir = repo_root / "docs" / "modules"
module_paths = sorted(module_dir.glob(DOC_GLOB))[:DOC_LIMIT]
print("documents:", len(module_paths))
for path in module_paths[:8]:
    print(" ", path.relative_to(repo_root))

course_chunks: list[Chunk] = []
start = time.perf_counter()
for path in module_paths:
    text = path.read_text(encoding="utf-8")
    course_chunks.extend(
        chunk_text(
            text,
            source=str(path.relative_to(repo_root)),
            chunk_size=COURSE_CHUNK_SIZE,
            chunk_overlap=COURSE_CHUNK_OVERLAP,
            metadata={"kind": "course-doc"},
        )
    )
elapsed = time.perf_counter() - start
print(f"chunks: {len(course_chunks):,} built in {elapsed:.2f}s")
show_chunks(course_chunks, limit=6)

documents: 23
  docs/modules/00-prerequisite-review.md
  docs/modules/01-autodiff.md
  docs/modules/02-tensors.md
  docs/modules/03-nn.md
  docs/modules/03b-training.md
  docs/modules/04-tokenizer.md
  docs/modules/05-embeddings.md
  docs/modules/06-language-models.md
chunks: 444 built in 0.01s


| i | source | span | chars | text |
| --- | --- | --- | --- | --- |
| 1 | docs/modules/00-prerequisite-review.md | 0:1400 | 1400 | # Module 00 — Prerequisite review\n\n> **Question this module answers:** *What do I need back in cache before buildin... |
| 2 | docs/modules/00-prerequisite-review.md | 1200:2600 | 1400 | sampling.\n### Computer science\n\n- **Functions and composition.** The whole course treats models as large composed ... |
| 3 | docs/modules/00-prerequisite-review.md | 2400:3800 | 1400 | g if derivatives, the chain rule, and "a computation as a graph" are already close at hand. Module 02 immediately mov... |
| 4 | docs/modules/00-prerequisite-review.md | 3600:5000 | 1400 | lains why logits, softmax, cross-entropy, and perplexity are the right language for prediction.\n- **ML workflow** ke... |
| 5 | docs/modules/00-prerequisite-review.md | 4800:6200 | 1400 | C)`: one row for each token ID, and one `C`-dimensional vector in each row. If token IDs have shape `(B, T)`, looking... |
| 6 | docs/modules/00-prerequisite-review.md | 6000:7400 | 1400 | ddings, attention, logits, and softmax; writing them down is the fastest way to catch a mistaken transpose, missing b... |

... 438 more chunks


In [18]:
course_embedder = HashEmbedder(dim=1024, ngram_range=(3, 5), seed=17)
start = time.perf_counter()
course_vectors = course_embedder.embed([chunk.text for chunk in course_chunks])
course_embed_seconds = time.perf_counter() - start

course_store = NumpyVectorStore(dim=course_embedder.dim)
course_store.add(course_chunks, course_vectors)
course_retriever = DenseRetriever(course_embedder, course_store)

print(course_store)
print(f"hash embedding wall time: {course_embed_seconds:.2f}s")

NumpyVectorStore(dim=1024, n=444)
hash embedding wall time: 1.05s


In [19]:
course_questions = [
    "Which module introduces retrieval augmented generation?",
    "What does Module 16 build?",
    "Which module adds tools?",
    "What is the deliverable for the capstone?",
]

for question in course_questions:
    print("=" * 80)
    print(question)
    show_retrieved(course_retriever.retrieve(question, k=3))

Which module introduces retrieval augmented generation?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.458 | docs/modules/17-rag.md |  to produce sentence embeddings" recipe. Most modern embedders (`nomic-embed`, `bge-base`, `mxbai-embed`) descend from this lineage. Read §3 (the architecture) and §4 (the loss ... |
| 2 | 0.431 | docs/modules/11-sampling.md | eneration" (2018).** The top-k paper, in the context of story generation. Older but worth reading; introduces the diversity-vs-quality framing that this whole module is about.\n... |
| 3 | 0.430 | docs/modules/17-rag.md | al-Augmented Generation for Knowledge-Intensive NLP Tasks" (NeurIPS 2020).** The paper that named "RAG." Read §3 (the model) — the joint training of retriever + generator is mor... |

What does Module 16 build?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.350 | docs/modules/17-rag.md | # Module 17 — Retrieval-augmented generation\n\n> **Question this module answers:** *How can the model use external knowledge it doesn't have memorized?*\n\n![Hero](17-rag/Modul... |
| 2 | 0.329 | docs/modules/05-embeddings.md | # Module 05 — Embeddings and positions\n\n> **Question this module answers:** *How do discrete symbols become meaning-like vectors?*\n\n![From token IDs to meaning-like vectors:... |
| 3 | 0.327 | docs/modules/16-inference.md | # Module 16 — Inference backends and production models\n\n> **Question this module answers:** *How do we get from "I built it" to "I can use it"?*\n\n![Module 16 summary diagram... |

Which module adds tools?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.329 | docs/modules/10-tinyllm.md | # Module 10 — Milestone: TinyLLM\n\n> **Question this module answers:** *Can I build a language model using the tools we learned?*\n\n![Pretraining the tiny GPT end-to-end: a ra... |
| 2 | 0.313 | docs/modules/19-agent.md | # Module 19 — Agent loops\n\n> **Question this module answers:** *How do we make the model pursue goals?*\n\n![Agent loop wrapped around a model backend and tool registry](19-ag... |
| 3 | 0.303 | docs/modules/20-capstone.md | ) COMPOSE CONTEXTUALIZED MESSAGE: history block + context block + the actual current question, in that order. (5) `agent.run(contextualized_message)` (Module 19) — planning step... |

What is the deliverable for the capstone?


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.476 | docs/modules/20-capstone.md | he conversation so the next turn's history is coherent. The `AssistantTurn.final_answer` stays `None` so machine-readable consumers (the eval harness) can detect the failure una... |
| 2 | 0.436 | docs/modules/17-rag.md | now" guard. Without it, models hallucinate when context is insufficient. With it, instruction-tuned models will (more often than not) actually abstain.\n\nNote what's *not* in t... |
| 3 | 0.436 | docs/modules/13-sft.md | # Module 13 — Instruction tuning (SFT)\n\n> **Question this module answers:** *How do we make the model follow requests?*\n\n![Module 13 on one page: the TinyShakespeare-pretrai... |

## Exercise 8 - Optional semantic embeddings with Ollama

Hash embeddings are lexical. If you ran `./prodlm.sh`, you should also have `nomic-embed-text` available through Ollama, and the next cell will build a semantic index over the same chunks. Skip it if you do not have Ollama running with `nomic-embed-text`.

In [20]:
OLLAMA_EMBED_MODEL = DEFAULT_OLLAMA_EMBED_MODEL
OLLAMA_EMBED_DIM = 768

ollama_retriever = None
ollama_embedder = OllamaEmbedder(OLLAMA_EMBED_MODEL, dim=OLLAMA_EMBED_DIM)
ollama_chunks = course_chunks
start = time.perf_counter()
ollama_vectors = ollama_embedder.embed([chunk.text for chunk in ollama_chunks])
elapsed = time.perf_counter() - start
ollama_store = NumpyVectorStore(dim=ollama_embedder.dim)
ollama_store.add(ollama_chunks, ollama_vectors)
ollama_retriever = DenseRetriever(ollama_embedder, ollama_store)
print(f"embedded {len(ollama_chunks)} chunks in {elapsed:.2f}s")

embedded 444 chunks in 26.89s


In [21]:
semantic_question = "What part of the course lets the assistant use information outside its weights?"

print("Hash retriever:")
show_retrieved(course_retriever.retrieve(semantic_question, k=8))

if ollama_retriever is not None:
    print("Ollama semantic retriever:")
    show_retrieved(ollama_retriever.retrieve(semantic_question, k=8))

Hash retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.534 | docs/modules/20-capstone.md | oughtful post-mortem possible. Plan accordingly.\n\n- **The from-scratch model is a comparison baseline, not the main backend.** Your Module 10 model may generate recognizable t... |
| 2 | 0.521 | docs/modules/20-capstone.md | n each chat() call" walks through the lifecycle in five steps — read Conversation; create fresh Scratchpad; run agent (lots of scratchpad activity); drop Scratchpad at the end o... |
| 3 | 0.513 | docs/modules/17-rag.md | ge collection of information in the form of a corpus (not necessarily the same corpus used in pretraining) and what information it selectively curates at [[16-inferance]] time t... |
| 4 | 0.508 | docs/modules/09-transformer-block.md | = 4 × embedding_dim` . The complete intermediate projection gives the GELU activation layer room to carve out nonlinear features. Because of this most of the parameters in a tra... |
| 5 | 0.508 | docs/modules/13-sft.md | # Module 13 — Instruction tuning (SFT)\n\n> **Question this module answers:** *How do we make the model follow requests?*\n\n![Module 13 on one page: the TinyShakespeare-pretrai... |
| 6 | 0.507 | docs/modules/20-capstone.md | integration layer. Earlier modules built the parts; this module decides what data flows between them, what state survives across calls, and what gets logged so you can debug beh... |
| 7 | 0.505 | docs/modules/16-inference.md | ll be about building the "shell" around it to make a useful assistant. Retrieval, tool usage, agent harnesses. These are all critical and complex components to make a system lik... |
| 8 | 0.503 | docs/modules/20-capstone.md | ) COMPOSE CONTEXTUALIZED MESSAGE: history block + context block + the actual current question, in that order. (5) `agent.run(contextualized_message)` (Module 19) — planning step... |

Ollama semantic retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.613 | docs/modules/16-inference.md | ll be about building the "shell" around it to make a useful assistant. Retrieval, tool usage, agent harnesses. These are all critical and complex components to make a system lik... |
| 2 | 0.607 | docs/modules/18-tools.md |  live web                │\n   │                                                                      │\n   │   • General assistant like tasks:                                  ... |
| 3 | 0.597 | docs/modules/15-evaluation.md |  Cover: what it can do (with eval numbers), what it cannot do (with eval numbers), how it fails (categorized), and how well-calibrated it is.\n- [ ] You can explain — out loud, ... |
| 4 | 0.584 | docs/modules/17-rag.md | t in its training data:                                │\n   │       - your private notes, your company's wiki, last week's news     │\n   │       - documents written after the ... |
| 5 | 0.573 | docs/modules/07-attention.md | interpretability angle. Builds intuition for what individual heads learn.\n\nOptional:\n\n- **Dao et al., "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awar... |
| 6 | 0.572 | docs/modules/20-capstone.md | nt.chat`.\n\n```bash\nsource .venv/bin/activate\n\npytest tests/test_assistant.py                          # all module-20 tests\npytest tests/test_assistant.py -x              ... |
| 7 | 0.569 | docs/modules/20-capstone.md | e loops.\n- Token-budget-aware summarization of old context.\n- Production sandboxing for arbitrary code execution.\n\nThe CLI and `/save` command are enough for this course: th... |
| 8 | 0.569 | docs/modules/18-tools.md | # Module 18 — Tool use\n\n> **Question this module answers:** *How can the model act outside itself?*\n\n![Module 18 on one page: a four-panel circus map of the tool-use loop. P... |

In [22]:
semantic_question = "Does SFT teach new capabilities or shape behavior?"

print("Hash retriever:")
show_retrieved(course_retriever.retrieve(semantic_question, k=8))

if ollama_retriever is not None:
    print("Ollama semantic retriever:")
    show_retrieved(ollama_retriever.retrieve(semantic_question, k=8))

Hash retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.393 | docs/modules/13-sft.md | ataset's surface regularities. This is both a feature and a bug. The exercises ask you to author the dataset yourself because the experience of writing 50 consistent examples is... |
| 2 | 0.376 | docs/modules/13-sft.md |  prediction over TinyShakespeare prose and continues a question prompt as if it were more prose. After SFT on 50 hand-authored (instruction, response) pairs — rendered through a... |
| 3 | 0.374 | docs/modules/09b-pretraining.md | vior selection\n\nThe corpus is not just fuel for the optimizer. It is the behavior distribution the model is asked to imitate. TinyShakespeare teaches character names, stage di... |
| 4 | 0.374 | docs/modules/12-scaling.md | (2022, "Emergent Abilities of Large Language Models"). The list included three-digit arithmetic, multi-step word problems, instruction following.\n\nSchaeffer et al. (2023, "Are... |
| 5 | 0.372 | docs/modules/12-scaling.md | ot `2 · V · D`.\n\n![Anatomy of a transformer's parameter count. Inside one block: self-attention's four projections total about 4·D²; the feedforward network with 4D inner widt... |
| 6 | 0.370 | docs/modules/14-dpo.md |  [[03b-training]] \n	* A post-SFT instruct model from the [[13-sft.md]] exercise notebook\n\n---\n## Where this fits in\n\nAfter Module 13 your SFT'd model produces well-formatt... |
| 7 | 0.370 | docs/modules/00-prerequisite-review.md | string and ask a coding agent for help. Blank answers are skipped rather than counted wrong.\n\n1. **Shape trace.** Follow token IDs through embedding lookup and projection to v... |
| 8 | 0.365 | docs/modules/13-sft.md | # Module 13 — Instruction tuning (SFT)\n\n> **Question this module answers:** *How do we make the model follow requests?*\n\n![Module 13 on one page: the TinyShakespeare-pretrai... |

Ollama semantic retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.648 | docs/modules/13-sft.md |  prediction over TinyShakespeare prose and continues a question prompt as if it were more prose. After SFT on 50 hand-authored (instruction, response) pairs — rendered through a... |
| 2 | 0.622 | docs/modules/13-sft.md | nt-turn-ends is.\n- [ ] You can explain — out loud, without notes — why "data quality dominates data quantity" applies more strongly at toy scale than at production scale.\n- [ ... |
| 3 | 0.621 | docs/modules/13-sft.md | lity problem, not a format problem. Module 15 returns  │\n   │   to it as the headline failure mode of toy-scale models.      │\n   └────────────────────────────────────────────... |
| 4 | 0.616 | docs/modules/13-sft.md | tes confidently — alignment teaches FORMAT, not TRUTH (the truth/calibration question waits for Modules 14–15). A bottom strip captions the headline: "SFT changes behavior, not ... |
| 5 | 0.614 | docs/modules/13-sft.md | ng small but directed *post-training* updates to the pretrained model.\n\n*Supervised fine-turning (SFT)* is the first layer of post training in a modern LLM stack. It is primar... |
| 6 | 0.608 | docs/modules/13-sft.md | elm.sh` to setup the model we'll use for post-training\n\n---\n## Where this fits in\n\nAfter Module 10 you should have at least one model that's completed pretraining. This is ... |
| 7 | 0.608 | docs/modules/14-dpo.md |  [[03b-training]] \n	* A post-SFT instruct model from the [[13-sft.md]] exercise notebook\n\n---\n## Where this fits in\n\nAfter Module 13 your SFT'd model produces well-formatt... |
| 8 | 0.607 | docs/modules/14-dpo.md | # Module 14 — Preference tuning (DPO)\n\n> **Question this module answers:** *Why is the model helpful, polite, or stylistically consistent?*\n\n![DPO at a glance: a pretrained-... |

In [23]:
semantic_question = "How can inference be optimized for performance efficiency?"

print("Hash retriever:")
show_retrieved(course_retriever.retrieve(semantic_question, k=8))

if ollama_retriever is not None:
    print("Ollama semantic retriever:")
    show_retrieved(ollama_retriever.retrieve(semantic_question, k=8))

Hash retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.410 | docs/modules/16-inference.md | antization ladder — more model, less memory. Lower precision means fewer bits per weight, smaller checkpoint, more model fits on a laptop. A vertical "ladder" shows precision st... |
| 2 | 0.406 | docs/modules/14-dpo.md | comparison prompts.\n\n1. **Hand-author preference data.** Create chosen/rejected examples around simple behavior preferences.\n2. **Run DPO.** Train policy vs reference and com... |
| 3 | 0.386 | docs/modules/16-inference.md | tive use (25–40 tok/s on M2 16GB).\n\nIf you have ≥ 32 GB, switching to `llama3.1:8b` is worth it for Module 19 (agent loops) — the 8B handles tool-calling significantly better ... |
| 4 | 0.378 | docs/modules/13-sft.md |  prediction over TinyShakespeare prose and continues a question prompt as if it were more prose. After SFT on 50 hand-authored (instruction, response) pairs — rendered through a... |
| 5 | 0.377 | docs/modules/16-inference.md | ing Large Language Model Decoding with Speculative Sampling" (2023).** Independent invention of the speculative-decoding idea, slightly different framing.\n\n## Deliverable chec... |
| 6 | 0.376 | docs/modules/09b-pretraining.md | vior selection\n\nThe corpus is not just fuel for the optimizer. It is the behavior distribution the model is asked to imitate. TinyShakespeare teaches character names, stage di... |
| 7 | 0.374 | docs/modules/15-evaluation.md | e different success criteria. Pick the one that matches the task.\n- **Substring matches can lie.** Searching for `"no"` also matches `"snow"`. Use specific references or strict... |
| 8 | 0.373 | docs/modules/17-rag.md | # Module 17 — Retrieval-augmented generation\n\n> **Question this module answers:** *How can the model use external knowledge it doesn't have memorized?*\n\n![Hero](17-rag/Modul... |

Ollama semantic retriever:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.703 | docs/modules/16-inference.md | th-`T+1` matrix).\n\nTotal: `O(T)` per step → `O(T²)` for the whole generation, but with a ~10× smaller constant. In practice, KV-cached inference is 5–20× faster than the naive... |
| 2 | 0.683 | docs/modules/16-inference.md | antization ladder — more model, less memory. Lower precision means fewer bits per weight, smaller checkpoint, more model fits on a laptop. A vertical "ladder" shows precision st... |
| 3 | 0.668 | docs/modules/16-inference.md |                                                           │\n   │     w  ≈ scale * round((w_fp16 - bias) / scale)                       │\n   │   where (scale, bias) are compute... |
| 4 | 0.667 | docs/modules/16-inference.md | branch prediction is to microprocessors*\n\nSpeculative decoding runs a small "draft" model (e.g., a 1B-param sibling of a 70B model) ahead of the main model, generating several... |
| 5 | 0.666 | docs/modules/16-inference.md | ing Large Language Model Decoding with Speculative Sampling" (2023).** Independent invention of the speculative-decoding idea, slightly different framing.\n\n## Deliverable chec... |
| 6 | 0.664 | docs/modules/16-inference.md | –20× speedup at production context lengths.*\n\nUnlike quantization, KV caching is pretty close to a free lunch. At least on larger models and context windows. It depends on the... |
| 7 | 0.662 | docs/modules/16-inference.md |                                                                     │\n   └──────────────────────────────────────────────────────────────────────┘\n```\n\nLatency is the user-fa... |
| 8 | 0.656 | docs/modules/17-rag.md | e question, search the index for the closest chunks, and put those chunks into the prompt before calling the inference backend.\n\nThe important distinction is that retrieval do... |

## Exercise 9 - Optional live RAG with model selection

The default live backend is ProdLM. To test a course-trained model instead, set `MODEL_SELECTION = "course"` for the strongest course artifact, or set it to a base artifact name like `"TinyLLM-30M"`; the loader will prefer `-DPO`, then `-SFT`, then the base artifact.

In [24]:
MODEL_SELECTION = "ProdLM"  # "ProdLM", "course", or an artifact base/name such as "TinyLLM-30M"
PRODLM_MODEL_ID = None  # optional Ollama tag override when MODEL_SELECTION == "ProdLM"
LIVE_DEVICE = "auto"
LIVE_TORCH_DTYPE = "float16"
USE_SEMANTIC_RETRIEVER_IF_AVAILABLE = True

rag_backend = None
try:
    rag_backend = load_selected_backend(
        MODEL_SELECTION,
        repo_root=repo_root,
        prodlm_model_id=PRODLM_MODEL_ID,
        device=LIVE_DEVICE,
        torch_dtype=LIVE_TORCH_DTYPE,
        required=False,
    )
    if rag_backend is None:
        print("No live backend loaded. Run ./prodlm.sh or choose an available artifact.")
    else:
        print("loaded:", rag_backend.info)
except Exception as exc:
    print(f"Live backend unavailable: {type(exc).__name__}: {exc}")

live_retriever = ollama_retriever if (USE_SEMANTIC_RETRIEVER_IF_AVAILABLE and ollama_retriever is not None) else course_retriever
print("retriever:", live_retriever)

loaded: BackendInfo(name='prodlm', model_id='llama3.2:3b', extra={'base_url': 'http://localhost:11434', 'configured_name': 'ProdLM'})
retriever: DenseRetriever(embedder=OllamaEmbedder(model_id='nomic-embed-text', dim=768, base_url='http://localhost:11434'), store=NumpyVectorStore(dim=768, n=444))


In [25]:
live_pipeline = None
if rag_backend is not None:
    live_pipeline = RAGPipeline(live_retriever, rag_backend)
else:
    print("No live backend loaded.")


def ask_rag(question: str, *, k: int = 4, max_new_tokens: int = 220):
    if live_pipeline is None:
        print("No live RAG pipeline loaded.")
        return None
    try:
        answer = live_pipeline.answer(
            question,
            k=k,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
        )
    except Exception as exc:
        print(f"RAG call failed: {type(exc).__name__}: {exc}")
        return None
    show_rag_answer(answer)
    return answer

In [26]:
live_question = "What does Module 17 teach, and why does retrieval quality matter?"
live_answer = ask_rag(live_question, k=8)

Based on the provided context, Module 17 teaches Retrieval-augmented generation (RAG), which is a method for giving a model access to information it doesn't have memorized. RAG works by chunking a corpus, embedding each chunk, storing the vectors, embedding a query, retrieving the top-k chunks by cosine similarity, and then splicing them into a citation-formatted prompt.

Retrieval quality matters because it dominates RAG quality. Improving the retriever buys more than improving the model, and retrieval quality is crucial for the success of RAG.

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.681 | docs/modules/05-embeddings.md | ck later: Module 17 retrieval\nalso ranks text chunks by vector similarity.\n\n## How to run the tests\n\nTests live in `tests/test_embeddings.py`. Construction tests pass from ... |
| 2 | 0.681 | docs/modules/17-rag.md | ises\n\nOpen the working notebook with `./notebook.sh 17` (or `./notebook.sh 17 --fresh` to reset from the clean scaffold). The notebook carries the exact indexing, retrieval, a... |
| 3 | 0.637 | docs/modules/17-rag.md | ion" (EMNLP 2023).** The "decide DURING generation whether to retrieve" angle — a step toward Module 19's agent loop. The model can issue mid-generation "retrieve again" request... |
| 4 | 0.636 | docs/modules/17-rag.md |  to produce sentence embeddings" recipe. Most modern embedders (`nomic-embed`, `bge-base`, `mxbai-embed`) descend from this lineage. Read §3 (the architecture) and §4 (the loss ... |
| 5 | 0.636 | docs/modules/17-rag.md | # Module 17 — Retrieval-augmented generation\n\n> **Question this module answers:** *How can the model use external knowledge it doesn't have memorized?*\n\n![Hero](17-rag/Modul... |
| 6 | 0.635 | docs/modules/13-sft.md | tes confidently — alignment teaches FORMAT, not TRUTH (the truth/calibration question waits for Modules 14–15). A bottom strip captions the headline: "SFT changes behavior, not ... |
| 7 | 0.626 | docs/modules/13-sft.md | nt-turn-ends is.\n- [ ] You can explain — out loud, without notes — why "data quality dominates data quantity" applies more strongly at toy scale than at production scale.\n- [ ... |
| 8 | 0.626 | docs/modules/16-inference.md | ing Large Language Model Decoding with Speculative Sampling" (2023).** Independent invention of the speculative-decoding idea, slightly different framing.\n\n## Deliverable chec... |

metadata: {'k': 8, 'n_retrieved': 8, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}


In [27]:
live_question = "What are the two parts of a transformer?"
live_answer = ask_rag(live_question, k=8)

Based on the provided context, I can answer the following questions:

1. What is the main difference between attention and the feed-forward network (FFN) in a transformer block?

According to [5], "The FFN is just a 1-hidden layer MLP, like the ones we trained in Module 3. It uses a slightly version of ReLU (GELU) as the activation layer: ... The most important thing to keep in mind is that the FFN is *per position*. There is no mixing between the token positions."

In other words, attention mixes information across tokens, while the FFN processes each token independently.

2. What is the purpose of residual connections in a transformer block?

According to [2], "Residual connections are how we 'stack' layers of transformer blocks... The intuition has two complementary flavors: ... Residual-stream view. Think of `x` as a 'communication bus' through the layers of the network."

In essence, residual connections allow the model to stack multiple layers without losing information, by addin

| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.629 | docs/modules/09-transformer-block.md | = 4 × embedding_dim` . The complete intermediate projection gives the GELU activation layer room to carve out nonlinear features. Because of this most of the parameters in a tra... |
| 2 | 0.627 | docs/modules/09-transformer-block.md |                               ▼\n                                                  (B, T, D)\n```\n\nTwo sublayers. Each sublayer is "normalize → transform → add to residual." T... |
| 3 | 0.599 | docs/modules/09-transformer-block.md | # Module 09 — The transformer block\n\n> **Question this module answers:** *How do we compose attention and "thinking"?*\n\n![One transformer block, drawn as two pre-norm sublay... |
| 4 | 0.555 | docs/modules/08-multi-head-attention.md | fter concatenating the H per-head outputs, you have a `(B, T, D)` tensor in which the first `head_dim` channels came from head 0, the next from head 1, and so on. Without a lear... |
| 5 | 0.547 | docs/modules/09-transformer-block.md | percent faster, no quality loss in practice.\n- **Press et al., "Using the Output Embedding to Improve Language Models" (2017).** The case for tied input/output embeddings — sav... |
| 6 | 0.543 | docs/modules/09-transformer-block.md |  dramatically less stable.\n\n- **Forgetting the residual.** Writing `x = self.attn(self.ln1(x))` instead of `x = x + self.attn(self.ln1(x))`. The model becomes untrainable past... |
| 7 | 0.540 | docs/modules/09b-pretraining.md | # Module 09B — Pretraining\n\n> **Question this module answers:** *How do we learn from text?*\n\n![Multi-position targets in three steps: sample a (B, T) window from the token ... |
| 8 | 0.538 | docs/modules/09-transformer-block.md |             ▲\n         └─── residual stream ───┘    ← residual flows past LN\n\n  Post-norm pipeline (Vaswani 2017):\n\n     x ──┬─► sublayer ──► + ──► LN\n         │          ... |

metadata: {'k': 8, 'n_retrieved': 8, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}


In [35]:
live_question = "How does AdamW scale gradients?"
live_answer = ask_rag(live_question, k=8)

According to the context, AdamW scales gradients by dividing the raw gradient by a term that includes the second moment of the gradient (v_hat) and an epsilon value. This is done to prevent large historical gradients from shrinking the effective step size for all parameters, which would make it difficult for the model to adapt to different parts of the parameter space.

The formula used in AdamW is:

```text
param <- param - lr * m_hat / (sqrt(v_hat) + eps)
```

This formula adapts the update per tensor element, allowing the model to scale gradients differently based on their magnitude.

Sources:


| rank | score | source | text |
| --- | --- | --- | --- |
| 1 | 0.725 | docs/modules/03b-training.md | w the step is scaled. \n\nAdamW still has a global `lr`, but the actual update is adapted per tensor element:\n\n```text\nparam <- param - lr * m_hat / (sqrt(v_hat) + eps)\n```\... |
| 2 | 0.725 | docs/modules/03b-training.md |  Vanilla SGD uses one global step size for every parameter, which often stalls on large models. Deep models occasionally produce gradient spikes; which can wreck an entire the r... |
| 3 | 0.716 | docs/modules/03b-training.md | l mode, RNG control, and activation scaling. For LLM pretraining, dropout is often small or zero. Worth knowing by name, but not worth spending a build week on here.\n\n## Conce... |
| 4 | 0.703 | docs/modules/03b-training.md | omplex architecture, different parts of the architecture live on different gradient scales.\n\nSGD uses one global `lr`:\n\n```text\nparam <- param - lr * grad\n```\n\nAdamW kee... |
| 5 | 0.703 | docs/modules/01-autodiff.md | erns.\n\n![Gradients](01-autodiff/Module01-Gradients.png)\n*What makes neural nets special?*\n\nFor this course, we do not start with gradients because every student needs to be... |
| 6 | 0.691 | docs/modules/03b-training.md | .](03b-training/Module03b-Norms.png)\n*A norm is just a measurement (panels 1–2), and clipping is the response when that measurement is too large (panel 3). Clipping multiplies ... |
| 7 | 0.672 | docs/modules/12-scaling.md |  off model size against dataset size).\n\n```\n   Validation loss\n        ▲\n        │  ●  (StoryLM-1M)\n        │\n        │      ●  (StoryLM-5M)\n        │\n        │        ... |
| 8 | 0.667 | docs/modules/03b-training.md | ip is visible:\n\n```text\nupdate = -lr * grad\n```\n\nIf `lr` is too small, the loss may be moving in the right direction but too slowly for your compute budget. If `lr` is too... |

metadata: {'k': 8, 'n_retrieved': 8, 'backend_name': 'prodlm', 'backend_model_id': 'llama3.2:3b'}


## Exercise 10 - Failure-mode probes

Good RAG work includes negative examples. You want to know when the model refuses, when retrieval misses, and when the generator invents facts anyway.

In [ ]:
failure_questions = [
    {"bucket": "answerable", "question": "Which module adds tools to the assistant?"},
    {"bucket": "paraphrase", "question": "Where in the course does the model get external memory?"},
    {"bucket": "unanswerable", "question": "What is the population of Pluto according to these course docs?"},
]

probe_rows = []
for item in failure_questions:
    retrieved = live_retriever.retrieve(item["question"], k=3)
    probe_rows.append(
        {
            "bucket": item["bucket"],
            "question": item["question"],
            "top source": retrieved[0].chunk.source if retrieved else "",
            "top score": f"{retrieved[0].score:.3f}" if retrieved else "",
            "top text": short(retrieved[0].chunk.text, 130) if retrieved else "",
        }
    )

display(Markdown(markdown_table(probe_rows, ["bucket", "question", "top source", "top score", "top text"])))

Now run the same probes through the live model. Requires a live backend; the cell prints a friendly skip message per question if no live RAG pipeline is loaded.

In [ ]:
for item in failure_questions:
    print("=" * 80)
    print(item["bucket"], "-", item["question"])
    ask_rag(item["question"], k=3, max_new_tokens=180)

## Exercise 11 - Persist an index extension

The course package keeps `NumpyVectorStore` in memory, but persisting an index is a natural extension. This cell gives a small starter shape you can adapt for the postmortem or an optional script.

In [ ]:
def save_store_snapshot(store: NumpyVectorStore, path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    np.save(path / "vectors.npy", store.vectors)
    chunks_json = [
        {
            "text": chunk.text,
            "source": chunk.source,
            "start": chunk.start,
            "end": chunk.end,
            "metadata": chunk.metadata,
        }
        for chunk in store.chunks
    ]
    (path / "chunks.json").write_text(json.dumps(chunks_json, indent=2), encoding="utf-8")


def load_store_snapshot(path: Path, *, dim: int) -> NumpyVectorStore:
    vectors = np.load(path / "vectors.npy")
    chunks_data = json.loads((path / "chunks.json").read_text(encoding="utf-8"))
    chunks = [Chunk(**item) for item in chunks_data]
    store = NumpyVectorStore(dim=dim)
    store.add(chunks, vectors)
    return store

snapshot_dir = repo_root / "data" / "module17-rag" / "hash-index-snapshot"
save_store_snapshot(course_store, snapshot_dir)
round_trip_store = load_store_snapshot(snapshot_dir, dim=course_embedder.dim)
print(round_trip_store)
print("same vectors:", np.allclose(course_store.vectors, round_trip_store.vectors))
print("same first chunk:", course_store.chunks[0] == round_trip_store.chunks[0])

## Postmortem notes

Write `docs/rag-postmortem.md` in 3-4 paragraphs. Cover:

- What you indexed: corpus, chunk size, overlap, embedder, vector count.
- What worked: question types where retrieval reliably surfaced the right chunk.
- Where it broke: chunking, embedding, retrieval, prompt, or model refusal/hallucination.
- What you would build next: hybrid retrieval, re-ranking, smarter chunking, larger embedder, or a better eval set.